# Restaurant Clustering

Builds the clustering outputs used by the dashboard:
- `data_output/reviews.csv`: review-level text corpus with cluster labels
- `data_output/clustering_results.csv`: restaurant-level cluster assignments with 2D coordinates


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from umap import UMAP
import scipy.sparse as sp

from sklearn.preprocessing import StandardScaler

BASE_DIR = Path("..")
KOL_PATH = BASE_DIR / "data" / "kol" / "KOL_Posts.csv"
PLACES_PATH = BASE_DIR / "_1_eda" / "data_output" / "places_api_new_results.csv"
REVIEWS_PATH = BASE_DIR / "_3_marketing" / "data_output" / "restaurant_reviews.parquet"
OUTPUT_DIR = Path("data_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GMV_VIEW = BASE_DIR / "_5_GA_data" / "data_output" / "gmv" / "gmv_view.parquet" # New clustering to be based on this GMV data

kol_posts = pd.read_csv(KOL_PATH) # KOL posts
places = pd.read_csv(PLACES_PATH) # Places API results
restaurant_reviews = pd.read_parquet(REVIEWS_PATH) # Restaurant reviews
gmv_view = pd.read_parquet(GMV_VIEW) # GMV data

kol_posts # 923 rows × 35 columns
places # 2392 rows × 12 columns
restaurant_reviews # 9674 rows × 15 columns
gmv_view # 17319 rows × 16 columns

restaurant_reviews

/Users/jadentyh/Desktop/IS455/Project/OPE/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,input_restaurant_name,pulled_at_utc,source,place_id,matched_name,review_id,author_name,author_url,profile_photo_url,rating,language,review_text,relative_time_description,review_time_unix,review_time_utc
0,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1770211654_NATTAWA...,NATTAWAT RATTANAPONGPRAKIT,https://www.google.com/maps/contrib/1024293807...,https://lh3.googleusercontent.com/a-/ALV-UjVOX...,5,en-US,We ate it all before I could take a picture! T...,a week ago,1770211654,2026-02-04T13:27:34+00:00
1,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1770021873_Mizutan...,Mizutani Koichi,https://www.google.com/maps/contrib/1102947333...,https://lh3.googleusercontent.com/a-/ALV-UjU8O...,1,en-US,The Happy Hour menu does not include draft bee...,2 weeks ago,1770021873,2026-02-02T08:44:33+00:00
2,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1769797681_Patpoom...,Patpoom MSoulScent,https://www.google.com/maps/contrib/1168736595...,https://lh3.googleusercontent.com/a-/ALV-UjVEH...,5,en-US,"The staff are nice, friendly, and polite.\nThe...",2 weeks ago,1769797681,2026-01-30T18:28:01+00:00
3,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1769692334_Jesada ...,Jesada Bobpahow,https://www.google.com/maps/contrib/1041664502...,https://lh3.googleusercontent.com/a-/ALV-UjVw9...,5,en-US,"Delicious food, great atmosphere.",2 weeks ago,1769692334,2026-01-29T13:12:14+00:00
4,Melody Bangkok,2026-02-16T15:06:12.727968+00:00,google_places,ChIJ-6r24ICf4jARC6jtWZYj7S8,Melody Bangkok,ChIJ-6r24ICf4jARC6jtWZYj7S8_1769692169_อรทัย ห...,อรทัย หลงทัพไทย,https://www.google.com/maps/contrib/1091362972...,https://lh3.googleusercontent.com/a/ACg8ocLqF_...,5,en-US,"The taste is good, the food is delicious.",2 weeks ago,1769692169,2026-01-29T13:09:29+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9669,Okuna Sushi Jungceylon (Phuket),2026-02-16T15:09:11.636531+00:00,google_places,ChIJQdyb6xg7UDARIxK_es9pIF4,Okuna Sushi Jungceylon (Phuket),ChIJQdyb6xg7UDARIxK_es9pIF4_1771235001_Jy S,Jy S,https://www.google.com/maps/contrib/1038710619...,https://lh3.googleusercontent.com/a-/ALV-UjVs6...,1,en-US,Three people went and ordered a variety of dis...,in the last week,1771235001,2026-02-16T09:43:21+00:00
9670,Okuna Sushi Jungceylon (Phuket),2026-02-16T15:09:11.636531+00:00,google_places,ChIJQdyb6xg7UDARIxK_es9pIF4,Okuna Sushi Jungceylon (Phuket),ChIJQdyb6xg7UDARIxK_es9pIF4_1770991294_Ni. Sha.,Ni. Sha.,https://www.google.com/maps/contrib/1030438452...,https://lh3.googleusercontent.com/a/ACg8ocIpyO...,1,en-US,Absolutely terrible! Warm sushi! Not recommended.,in the last week,1770991294,2026-02-13T14:01:34+00:00
9671,Okuna Sushi Jungceylon (Phuket),2026-02-16T15:09:11.636531+00:00,google_places,ChIJQdyb6xg7UDARIxK_es9pIF4,Okuna Sushi Jungceylon (Phuket),ChIJQdyb6xg7UDARIxK_es9pIF4_1770108077_Serap C...,Serap Civan,https://www.google.com/maps/contrib/1021855615...,https://lh3.googleusercontent.com/a/ACg8ocK57D...,5,NaN,,a week ago,1770108077,2026-02-03T08:41:17+00:00
9672,Okuna Sushi Jungceylon (Phuket),2026-02-16T15:09:11.636531+00:00,google_places,ChIJQdyb6xg7UDARIxK_es9pIF4,Okuna Sushi Jungceylon (Phuket),ChIJQdyb6xg7UDARIxK_es9pIF4_1769696963_Gökhan ...,Gökhan KILIÇ,https://www.google.com/maps/contrib/1066853741...,https://lh3.googleusercontent.com/a-/ALV-UjXjK...,4,NaN,,2 weeks ago,1769696963,2026-01-29T14:29:23+00:00


## 1. Build the Restaurant Text Corpus


In [2]:
# 1. Aggregate KOL content with ID
kol_aggregated = (
    kol_posts.groupby(["Restaurant Code", "Restaurant Name"])["Content"]
    .apply(lambda s: " ".join(s.dropna().astype(str)))
    .reset_index()
    .rename(columns={"Restaurant Name": "restaurant name", "Content": "kol_text", "Restaurant Code": "restaurant_id"})
)

# 2. Prepare Google Places data with ID
places = places.copy()
places["google_text"] = (
    places["input_string"].fillna("").astype(str)
    + " "
    + places["raw_types"].fillna("").astype(str).str.replace(",", " ")
    + " "
    + places["Cuisine"].fillna("").astype(str)
)

place_text = (
    places[["input_string", "google_text"]]
    .rename(columns={"input_string": "restaurant name"})
    .drop_duplicates(["restaurant name"])
)

# 3. Aggregate Reviews with ID
review_text = (
    restaurant_reviews.groupby(["input_restaurant_name"])["review_text"]
    .apply(lambda s: " ".join(s.dropna().astype(str)))
    .reset_index()
    .rename(columns={"input_restaurant_name": "restaurant name"})
)

# 4. Final Merge on ID (The Primary Key)
# We merge on restaurant_id to ensure absolute accuracy
reviews = (
    review_text
    .merge(place_text, on="restaurant name", how="left", suffixes=("", "_drop"))
    .merge(kol_aggregated, on="restaurant name", how="left", suffixes=("", "_drop"))
)

# Clean up duplicate name columns from the merge if they exist
reviews = reviews.drop(columns=[c for c in reviews.columns if c.endswith("_drop")])

# 5. Combine Text
reviews["raw_text"] = (
    reviews["review_text"].fillna("").astype(str)
    + " "
    + reviews["google_text"].fillna("").astype(str)
    + " "
    + reviews["kol_text"].fillna("").astype(str)
).str.replace(r"\s+", " ", regex=True).str.strip()

reviews = reviews[reviews["raw_text"].str.len() > 0].copy()

## 2. Clean and Vectorize Text


In [3]:
redundant_words = [
    "point_of_interest",
    "establishment",
    "general",
    "food establishment",
    "restaurant point_of_interest",
    "food point_of_interest",
    "point_of_interest establishment",
    "establishment general",
    "restaurant food",
    "meal_delivery",
    "meal_takeaway",
    "lodging",
    "tourist_attraction",
    "night_club",
    "shopping_mall",
    "bangkok",
    "sukhumvit",
    "singapore",
    "thai",
    "baht",
    "order",
    "ordered",
    "dishes",
    "fried",
    "sauce",
    "spicy",
    "time",
    "eat",
    "meat",
    "restaurant",
    "soup",
    "super",
    "really",
    "terrible",
    "floor",
    "rice",
    "pork",
    "beef",
    "chicken",
    "crab",
]

def clean_text(text: str) -> str: # Clean text by removing redundant words and non-alphabetic characters
    cleaned = str(text).lower()
    for phrase in sorted(redundant_words, key=len, reverse=True):
        cleaned = cleaned.replace(phrase, " ")
    cleaned = re.sub(r"[^a-z\s]", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

reviews["clean_text"] = reviews["raw_text"].apply(clean_text)
reviews = reviews[reviews["clean_text"].str.len() > 0].copy()

vectorizer = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2,
)
text_vectors = vectorizer.fit_transform(reviews["clean_text"])
print(reviews.shape)
reviews.columns

(1946, 7)


Index(['restaurant name', 'review_text', 'google_text', 'restaurant_id',
       'kol_text', 'raw_text', 'clean_text'],
      dtype='str')

In [ ]:
## AGGREGATE GMV DATA

# 1. Aggregate GMV data to the restaurant level (All-Time Average)
gmv_agg = gmv_view.groupby('name', as_index=False).agg(
    avg_monthly_gmv=('monthly_gmv', 'mean'),
    avg_monthly_bookings=('monthly_bookings', 'mean'),
    avg_bookings_per_view=('bookings_per_view', 'mean'),
    avg_gmv_per_view=('gmv_per_view', 'mean'),
    avg_view_to_purchase=('view_to_purchase_rate', 'mean'),
    avg_rev_per_view=('revenue_per_view', 'mean'),
    avg_ga_add_to_cart_rate=('ga_add_to_cart_rate', 'mean'),
    avg_purchase_to_cart_rate=('purchase_to_cart_rate', 'mean')
)

# 2. Merge GMV features into the review-level dataframe
reviews = reviews.merge(gmv_agg, left_on='restaurant name', right_on='name', how='left')
reviews = reviews.drop(columns=['name']) # Drop the redundant 'name' column after merge

# 3. Handle missing data (for restaurants in reviews but not in GA)
gmv_cols = [
    'avg_monthly_gmv', 'avg_monthly_bookings', 
    'avg_gmv_per_view', 'avg_view_to_purchase', 'avg_rev_per_view'
]
reviews[gmv_cols] = reviews[gmv_cols].fillna(0)

# 4. Scale the GMV features
# TF-IDF values are between 0 and 1. GMV values are in the thousands. 
# We MUST scale them so they don't completely overpower the text data.
scaler = StandardScaler()
gmv_scaled = scaler.fit_transform(reviews[gmv_cols])

# Optional: Weight adjustment. 
# Since TF-IDF has 200 dimensions and GMV only has 5, we multiply the GMV 
# vectors by a weight factor so they have a meaningful impact on the clusters.
gmv_weight = 2.0
gmv_scaled = gmv_scaled * gmv_weight

# 5. Combine Text Vectors and GMV Vectors into a single feature matrix
# hstack efficiently combines the sparse text vectors with the dense GMV vectors into one feature set for clustering.
combined_features = sp.hstack([text_vectors, gmv_scaled])

print(f"Text vectors shape: {text_vectors.shape}")
print(f"Combined features shape: {combined_features.shape}")
print(f"Reviews + bookings data shape: {reviews.shape}")

Text vectors shape: (1946, 200)
Combined features shape: (1946, 205)
Reviews + bookings data shape: (1946, 15)


## 3. Cluster Reviews


In [5]:
cluster_themes = {
    0: "Bar & Alcoholic Drinks",
    1: "Good food and atmosphere",
    2: "Buffet & premium dining",
    3: "Excellent service",
    4: "Cafes, Coffee, Breakfast",
}

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
reviews["cluster"] = kmeans.fit_predict(text_vectors)
reviews["theme"] = reviews["cluster"].map(cluster_themes)
reviews.to_csv(OUTPUT_DIR / "reviews.csv", index=False)

reviews # 1946

,restaurant name,review_text,google_text,restaurant_id,kol_text,raw_text,clean_text,avg_monthly_gmv,avg_monthly_bookings,avg_bookings_per_view,avg_gmv_per_view,avg_view_to_purchase,avg_rev_per_view,avg_ga_add_to_cart_rate,avg_purchase_to_cart_rate,cluster,theme
0,&T modern Thai grill,-The restaurant is beautifully decorated with ...,NaN,NaN,NaN,-The restaurant is beautifully decorated with ...,the is beautifully decorated with a nice atmos...,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,NaN,NaN,3,Excellent service
1,'@RICE Restaurant by At Rice Resort (Nakhon Na...,"The drinks and snacks are delicious, good tast...",'@RICE Restaurant by At Rice Resort (Nakhon Na...,NaN,NaN,"The drinks and snacks are delicious, good tast...",the drinks and snacks are delicious good taste...,1271.666667,1.500000,0.060415,51.866411,0.072020,6.684808,0.089482,0.666667,1,Good food and atmosphere
2,100 Degrees Hotpot Central Chaengwattana,"Delicious, good price, great value. This was m...",100 Degrees Hotpot Central Chaengwattana resta...,NaN,NaN,"Delicious, good price, great value. This was m...",delicious good p gr value this was my first in...,1253.333333,2.666667,0.033640,15.810799,0.037186,2.097296,0.029309,1.166667,3,Excellent service
3,100 Degrees Hotpot Cosmo Bazaar,Kimchi soup is the best! So delicious. I've ea...,100 Degrees Hotpot Cosmo Bazaar restaurant poi...,NaN,NaN,Kimchi soup is the best! So delicious. I've ea...,kimchi is the best so delicious i ve en it man...,867.500000,1.750000,0.044276,21.535402,0.052261,3.110681,0.000000,NaN,3,Excellent service
4,123 ICHI NI SAN Sathorn Soi 1,"The food here is delicious, and they have gre...",123 ICHI NI SAN Sathorn Soi 1 japanese_restaur...,NaN,NaN,"The food here is delicious, and they have grea...",the food here is delicious and they have gr de...,1906.000000,1.000000,0.008324,15.798062,0.017514,3.325586,0.016095,0.936508,0,Bar & Alcoholic Drinks
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1941,sala ayutthaya staycation (Ayutthaya),A gem!\n\n2 nights / 2 breakfasts / 2 dinners...,sala ayutthaya staycation (Ayutthaya) hotel th...,NaN,NaN,A gem! 2 nights / 2 breakfasts / 2 dinners Pro...,a gem nights breakfasts dinners professional a...,4550.000000,1.000000,0.003754,16.928459,0.004213,2.317757,0.078338,0.054100,2,Buffet & premium dining
1942,sala bang pa-in Staycation (Ayutthaya),"This stunning resort hotel, like a paradise on...",sala bang pa-in Staycation (Ayutthaya) hotel r...,NaN,NaN,"This stunning resort hotel, like a paradise on...",this stunning resort hotel like a paradise on ...,13000.000000,1.333333,0.008732,106.673882,0.003175,5.447619,0.064602,0.044444,2,Buffet & premium dining
1943,อร่อย Together,Went to sing karaoke and was pleasantly surpri...,อร่อย Together restaurant point_of_interest fo...,NaN,NaN,Went to sing karaoke and was pleasantly surpri...,went to sing karaoke and was pleasantly surpri...,6750.000000,1.000000,0.000874,5.900350,0.003497,2.360140,0.045455,0.076923,3,Excellent service
1944,中国好味道 China Taste,Food is relatively less salty and more home co...,中国好味道 China Taste restaurant asian_restaurant ...,5899,Affordable Chinese Food at this more than a de...,Food is relatively less salty and more home co...,food is relatively less salty and more home co...,55.600000,1.000000,0.004445,0.246710,0.009582,1.617455,0.008197,0.333333,4,"Cafes, Coffee, Breakfast"


## 4. Build Restaurant-Level Assignments


In [6]:
reducer = UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=1.0,
    metric="cosine",
    random_state=42,
)
embeddings_2d = reducer.fit_transform(text_vectors.toarray())

review_points = reviews.copy()
review_points["x"] = embeddings_2d[:, 0]
review_points["y"] = embeddings_2d[:, 1]

def dominant_value(series: pd.Series):
    counts = series.value_counts()
    return counts.index[0] if len(counts) else pd.NA

def dominant_share(series: pd.Series) -> float:
    counts = series.value_counts(normalize=True)
    return float(counts.iloc[0]) if len(counts) else np.nan

clustering_results = (
    review_points.groupby("restaurant name", as_index=False)
    .agg(
        # Existing Cluster Logic
        restaurant_id=("restaurant_id", "first"),
        cluster=("cluster", dominant_value),
        theme=("theme", dominant_value),
        x=("x", "mean"),
        y=("y", "mean"),
        cluster_confidence=("cluster", dominant_share),
        review_count=("raw_text", "size"),
        # New Performance Metrics
        avg_monthly_gmv=("avg_monthly_gmv", "mean"),
        avg_monthly_bookings=("avg_monthly_bookings", "mean"),
        avg_ga_add_to_cart_rate=("avg_ga_add_to_cart_rate", "mean"),
        avg_gmv_per_view=("avg_gmv_per_view", "mean"),
        avg_view_to_purchase=("avg_view_to_purchase", "mean"),
        avg_rev_per_view=("avg_rev_per_view", "mean")
    )
    .sort_values(["cluster", "restaurant name"])
    .reset_index(drop=True)
)

clustering_results.to_csv(OUTPUT_DIR / "clustering_results.csv", index=False)
clustering_results

/Users/jadentyh/Desktop/IS455/Project/OPE/venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


,restaurant name,restaurant_id,cluster,theme,x,y,cluster_confidence,review_count,avg_monthly_gmv,avg_monthly_bookings,avg_ga_add_to_cart_rate,avg_gmv_per_view,avg_view_to_purchase,avg_rev_per_view
0,123 ICHI NI SAN Sathorn Soi 1,NaN,0,Bar & Alcoholic Drinks,6.820526,0.656272,1.0,1,1906.000000,1.000000,0.016095,15.798062,0.017514,3.325586
1,298 Nikuya Yakiniku,NaN,0,Bar & Alcoholic Drinks,5.901494,1.081181,1.0,1,12804.571429,5.000000,0.040853,28.150125,0.012766,3.969336
2,AKATO BKK,NaN,0,Bar & Alcoholic Drinks,8.117333,2.446491,1.0,1,105273.857143,60.714286,0.088713,9.217013,0.008129,1.453688
3,AKI Traditional Omakase,NaN,0,Bar & Alcoholic Drinks,6.775575,5.492903,1.0,1,39678.181818,3.090909,0.066803,171.292661,0.011665,16.116825
4,AV izakaya Lat Phrao 71,NaN,0,Bar & Alcoholic Drinks,8.923729,1.296137,1.0,1,1950.000000,1.000000,0.000000,20.117329,0.001805,0.314079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1940,Xin Tian Di Crowne Plaza Bangkok Lumpini Park,2483,4,"Cafes, Coffee, Breakfast",1.697965,5.923288,1.0,1,164573.307692,65.076923,0.089986,26.826450,0.019986,6.247722
1941,YOK Chinese Restaurant | The Emerald Hotel,6016,4,"Cafes, Coffee, Breakfast",2.758512,6.477816,1.0,1,44677.250000,10.750000,0.055706,42.166713,0.015995,5.806558
1942,Yakiniku Shoutaian 2nd Rich,3030,4,"Cafes, Coffee, Breakfast",0.699697,5.543721,1.0,1,80811.615385,13.384615,0.030396,26.945120,0.007575,3.952658
1943,Yumcha Teahouse Empire Tower,5830,4,"Cafes, Coffee, Breakfast",2.286265,6.202992,1.0,1,170511.000000,82.222222,0.087995,32.306297,0.024825,6.445689
